In [ ]:
# MATLAB section 1/1 for CovCollExamples: Test CovColl

# % Test CovColl
# MATLAB: close all;
# MATLAB: load CovariateSample.mat;
# MATLAB: cc=CovColl({position,force});
# MATLAB: figure; cc.plot; %plots all covariates and their components
#
# MATLAB: cc.getCov(1); %returns position;
# MATLAB: cc.getCov('Position');
# MATLAB: cc.getCov({'Position','Force'});
# MATLAB: cc.resample(200); %resamples both position and force
#
#
# MATLAB: cc.setMask({{'Position','x'},{'Force','f_y'}});
# MATLAB: figure; cc.plot; %plot only x and f_y;
#
# dataToMatrix
# setMaxTime
# setMinTime
# removeCovariate
# nActCovar
#

# Python translation bootstrap + execution for single-section helpfile.

# Topic: CovCollExamples
# Execution group: full
# Workflow family: signal
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/CovCollExamples.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "CovCollExamples"
FAMILY = "signal"
np.random.seed(2026)
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"CovCollExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"CovCollExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"CovCollExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"CovCollExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


# CovCollExamples: covariate collection queries, masking, and resampling.
from nstat.compat.matlab import Covariate, CovColl, History, nspikeTrain

t = np.arange(0.0, 5.0 + 0.001, 0.001)
position = Covariate(time=t, data=np.column_stack([np.exp(-t), np.sin(2.0 * np.pi * t), np.sin(2.0 * np.pi * t) ** 3]), name="Position", labels=["x", "y", "z"])
force = Covariate(time=t, data=np.column_stack([np.abs(np.sin(2.0 * np.pi * t)), np.abs(np.sin(2.0 * np.pi * t)) ** 2]), name="Force", labels=["f_x", "f_y"])
cc = CovColl([position, force]); cc.resample(200.0); cc.setMask(["Position", "Force"])
fig, axes = plt.subplots(1, 2, figsize=(10, 4)); plt.sca(axes[0]); cc.plot(); axes[0].set_title(f"{TOPIC}: resampled")

X, labels = cc.dataToMatrix(); cc.removeCovariate("Force"); n_after = cc.nActCovar()
history = History(bin_edges_s=np.array([0.0, 0.01, 0.03], dtype=float))
spikes = nspikeTrain(spike_times=np.sort(rng.random(25) * 0.5), t_start=0.0, t_end=0.5, name="tmp")
H = history.computeHistory(spikes.spike_times, np.arange(0.0, 0.5, 0.01))
axes[1].imshow(H.T, aspect="auto", origin="lower", cmap="magma"); axes[1].set_title("History basis")
plt.tight_layout(); plt.show()

assert H.ndim == 2 and H.shape[1] == history.n_bins
assert spikes.spike_times.size > 5
CHECKPOINT_METRICS = {"matrix_rows": float(X.shape[0]), "matrix_cols": float(X.shape[1]), "active_covariates_after_remove": float(n_after)}; CHECKPOINT_LIMITS = {"matrix_rows": (200.0, 2000.0), "matrix_cols": (4.0, 8.0), "active_covariates_after_remove": (1.0, 3.0)}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)
